# Hashtag maskelaging

In [ ]:
%pip install pandas

In [ ]:
import hyperspy.api as hs
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
import pandas as pd

### Laster inn alle zspy filene --> Brightfield

In [ ]:
#funksjon som laster inn zspy-fil og summerer diffrkasjonsmønsteret --> brightfield-bilde
def process_stem(file_path):
    s = hs.load(file_path, lazy=True)
    s_bs = s.sum(axis=(-1, -2))
    s_bs.compute()
    return s_bs

#bruker funksjonen på alle zspy-filene fra 002 - 023 = 22 filer
data_folder = Path("DPC_zspy")
files = sorted(data_folder.glob("*.zspy"))
original_images = {}

for file in files:
    print(f"Processing {file.name}")
    original_images[file.stem] = process_stem(file)

#Sjekker om filene er prosessert: 
print(f"Processed {len(images)} files")

### Regner ut affintransformasjon

Tar utgangspunkt i referansebilde 002 (når tilt = 0 grader, og diskene er sirkulære), finner karakteristiske punkter på dette bildet vha SIFT, og finner tilsvarende punkter i bildet som er tatt ved en ikke-0 tilt. Beregner utifra dette en affin-transformasjon, altså en transformasjonsmatrise, som beskriver sammenhengen mellom punktene.

In [ ]:
import numpy as np
from skimage.feature import SIFT, match_descriptors
from skimage.measure import ransac
from skimage.transform import AffineTransform

#Funksjonen som finner affintransformasjon med utgangspunkt i referansebilde 002
def compute_affine(ref_img, img, max_distance=20):

    # SIFT on reference image
    sift1 = SIFT(
    n_octaves=5,
    n_scales=4,
    c_dog=0.01,
    c_edge=15
    )
    sift1.detect_and_extract(ref_img)

    # SIFT on tilted image
    sift2 = SIFT(
    n_octaves=5,
    n_scales=4,
    c_dog=0.01,
    c_edge=15
    )
    sift2.detect_and_extract(img)

    # Match descriptors
    matches = match_descriptors(
        sift1.descriptors,
        sift2.descriptors,
        cross_check=True,
        max_ratio=0.8 #visstnok ofte mellom 0.7-0.85?
    )

    # Filter matches based on distance
    pts1 = sift1.keypoints[matches[:, 0]]
    pts2 = sift2.keypoints[matches[:, 1]]

    distances = np.linalg.norm(pts1 - pts2, axis=1)
    good_matches = matches[distances < max_distance]

    print("Original matches:", len(matches))
    print("Filtered matches:", len(good_matches))

    if len(good_matches) < 3:
        raise ValueError("Not enough matches for affine transformation")

    # Convert (row, col) to (x, y) --> nødvendig når vi skal bruke scikit.image
    src = sift1.keypoints[good_matches[:, 0]][:, ::-1]
    dst = sift2.keypoints[good_matches[:, 1]][:, ::-1]

    # RANSAC: velger de matchene som passer med en felles affin transformasjon (inliers) --> ganske heftig filter visstnok
    transform, inliers = ransac(
        (src, dst),
        AffineTransform,
        min_samples=3,
        residual_threshold=2,
        max_trials=1000,
        rng=0
    )

    if transform is None or np.sum(inliers) < 3:
        raise ValueError("RANSAC could not find a valid transform")

    print("RANSAC inliers:", np.sum(inliers))

    return transform, sift1, sift2, good_matches, inliers

Tester på bilde 003 :

In [ ]:
#Tester på bilde 003: 

from skimage.transform import warp

# Select images
ref_img = np.asarray(original_images["002"], dtype=float)
img = np.asarray(original_images["003"], dtype=float)

# Normalize images
ref_img = (ref_img - ref_img.min()) / (ref_img.max() - ref_img.min())
img = (img - img.min()) / (img.max() - img.min())


transform, sift1, sift2, good_matches, inliers = compute_affine(
    ref_img,
    img,
    max_distance=20
)

img_corrected = warp(
    img,
    inverse_map=transform,
    output_shape=ref_img.shape,
    preserve_range=True
)

print("Transformation matrix:")
print(transform.params)

import matplotlib.pyplot as plt
from skimage.feature import plot_matched_features

plt.figure(figsize=(15, 5))

# Reference
plt.subplot(1, 3, 1)
plt.imshow(ref_img, cmap="gray")
plt.title("Reference (002)")
plt.axis("off")

# Original
plt.subplot(1, 3, 2)
plt.imshow(img, cmap="gray")
plt.title("Original (003)")
plt.axis("off")

# Corrected
plt.subplot(1, 3, 3)
plt.imshow(img_corrected, cmap="gray")
plt.title("Corrected (003)")
plt.axis("off")

plt.tight_layout()
plt.show()

# Plot only RANSAC-approved SIFT matches
fig, ax = plt.subplots(figsize=(14, 8))

plot_matched_features(
    ref_img,
    img,
    keypoints0=sift1.keypoints,
    keypoints1=sift2.keypoints,
    matches=good_matches[inliers],
    ax=ax,
    only_matches=True
)
plt.title("SIFT matches: (002) and (003)")

plt.show()

Tester om overlapp mellom referanse og 003 ser greit ut: 

In [ ]:
from skimage.transform import warp
import matplotlib.pyplot as plt

img_corrected = warp(
    img,
    inverse_map=transform,
    output_shape=ref_img.shape,
    preserve_range=True
)

plt.figure(figsize=(8, 8))

plt.imshow(ref_img, cmap="Reds", alpha=0.5)
plt.imshow(img_corrected, cmap="Greens", alpha=0.5)

plt.title("Overlay: Reference (red) + Corrected (green)")
plt.axis("off")
plt.show()

### RMS error

RMS error for hvor godt affine transformation beskriver de matchende SIFT-punktene. Vi ønsker å undersøke hvor nær den transformerte posisjonen (prediksjonen) kommer den faktiske posisjonen til det matchede punktet. 


In [ ]:
src = sift1.keypoints[good_matches[inliers, 0]][:, ::-1]
dst = sift2.keypoints[good_matches[inliers, 1]][:, ::-1]

predicted = transform(src)

errors = np.linalg.norm(predicted - dst, axis=1)
rms_error = np.sqrt(np.mean(errors**2))

print(f"RMS registration error: {rms_error:.3f} pixels")

### Funksjon som utfører affintransformasjon på ett bilde:

1. Finner SIFT-punkter i referansebildet og bildet som skal korrigeres.
2. Matcher punktene mellom bildene.
3. Filtrerer bort matcher med avstand over 20 piksler.
4. Bruker RANSAC til å finne en robust affin transformasjon.
5. Beregner RMS-feilen for de godkjente matchene.
6. Returnerer det korrigerte bildet og informasjon om registreringen.

--> returnerer altså en dictionary, hvor "image" er av størst interesse (?)

In [ ]:
def register_image(ref_img, img, max_distance=20):

    # STEP 1: Find SIFT features in both images
    # Lower c_dog makes SIFT detect weaker contrast features.
    sift1 = SIFT(
        n_octaves=5,
        n_scales=4,
        c_dog=0.01,
        c_edge=15
    )
    sift1.detect_and_extract(ref_img)

    sift2 = SIFT(
        n_octaves=5,
        n_scales=4,
        c_dog=0.01,
        c_edge=15
    )
    sift2.detect_and_extract(img)

    # STEP 2: Match corresponding SIFT features
    matches = match_descriptors(
        sift1.descriptors,
        sift2.descriptors,
        cross_check=True,
        max_ratio=0.8
    )

    n_matches = len(matches)

    # STEP 3: Filter matches based on distance
    # Assumes the images are already approximately aligned.
    pts1 = sift1.keypoints[matches[:, 0]]
    pts2 = sift2.keypoints[matches[:, 1]]

    distances = np.linalg.norm(pts1 - pts2, axis=1)
    good_matches = matches[distances < max_distance]

    n_filtered = len(good_matches)

    # At least 3 non-collinear point pairs are needed
    # to estimate an affine transformation.
    if n_filtered < 3:
        raise ValueError("Not enough matches for affine transformation")

    # STEP 4: Convert coordinates from (row, col) to (x, y)
    src = sift1.keypoints[good_matches[:, 0]][:, ::-1]
    dst = sift2.keypoints[good_matches[:, 1]][:, ::-1]

    # STEP 5: Estimate affine transformation using RANSAC
    # RANSAC rejects matches that do not follow the same transformation.
    transform, inliers = ransac(
        (src, dst),
        AffineTransform,
        min_samples=3,
        residual_threshold=2,
        max_trials=1000,
        rng=0
    )

    if transform is None or np.sum(inliers) < 3:
        raise ValueError("RANSAC failed to find a valid transformation")

    n_inliers = int(np.sum(inliers))

    # STEP 6: Calculate RMS registration error
    # Compare transformed reference points with their matched positions.
    predicted = transform(src[inliers])

    errors = np.linalg.norm(predicted - dst[inliers], axis=1)
    rms_error = np.sqrt(np.mean(errors**2))

    # STEP 7: Apply affine transformation
    # Transform maps reference -> moving image coordinates.
    # warp uses this mapping to sample the moving image on
    # the reference image's pixel grid.
    img_corrected = warp(
        img,
        inverse_map=transform,
        output_shape=ref_img.shape,
        preserve_range=True
    )

    return {
        "image": img_corrected,
        "transform": transform,
        "n_matches": n_matches,
        "n_filtered": n_filtered,
        "n_inliers": n_inliers,
        "rms_error": rms_error
    }

### Utfører SEKVENSIELL affintransformasjon på alle bildene: 

Istedet for å utføre affintransformasjon fra hver av bildene direkte til referansebildet, kjører vi her en sekvensiell metode: 
AT på 003 -->002, deretter AT på 004-->003, og bruker den forrige AT -->002, deretter 005-->004, og bruker de 2 forrige AT... 
Ettersom det er mindre forskjeller mellom hver av nabobildene, vil dette forhåpentligvis gjøre det lettere å finne samsvarende nøkkelpunkter mellom de to bildene, og dermed en bedre estimering av affintransformasjonene. Det er derimot også større risiko mtp akummuleringsfeil...

In [ ]:
#Normalisering sørger for en felles intensitetsskala når karakteristiske punkter skal identifiseres --> klargjør bildene til affintransformasjon
def normalize_image(image):
    # Convert image to float
    image = np.asarray(image, dtype=float)

    # Normalize intensity values to [0, 1]
    min_val = image.min()
    max_val = image.max()

    if max_val == min_val:
        raise ValueError("Image has constant intensity")

    return (image - min_val) / (max_val - min_val)



from skimage.transform import warp, AffineTransform
import numpy as np
import pandas as pd

def register_all_images(images, reference="002", max_distance=20):

    # Sort images in numerical order
    names = sorted(images.keys(), key=int)
    names = names[names.index(reference):]

    # Normalize all images
    normalized_images = {
        name: normalize_image(images[name].data)
        for name in names
    }

    ref_img = normalized_images[reference]

    # Dictionaries for registered images and transformations
    registered_images = {}
    transformations = {}
    pairwise_transformations = {}

    # Store registration statistics
    results = []

    # Reference image needs no transformation
    registered_images[reference] = ref_img

    # Identity matrix for reference image
    cumulative_matrix = np.eye(3)
    transformations[reference] = AffineTransform(
        matrix=cumulative_matrix.copy()
    )

    results.append({
        "image": reference,
        "matches": np.nan,
        "filtered": np.nan,
        "inliers": np.nan,
        "rms_error": 0.0,
        "status": "reference"
    })

    previous_name = reference

    for name in names[1:]:

        print(f"Registering {name} to {previous_name}...")

        # Previous and current ORIGINAL normalized images
        previous_img = normalized_images[previous_name]
        img = normalized_images[name]

        # STEP 1: SIFT + RANSAC between neighboring images
        result = register_image(
            previous_img,
            img,
            max_distance=max_distance
        )

        # STEP 2: Store pairwise affine transformation
        pairwise_transform = result["transform"]
        pairwise_transformations[name] = pairwise_transform

        # STEP 3: Combine with all previous transformations
        cumulative_matrix = (
            pairwise_transform.params @ cumulative_matrix
        )

        cumulative_transform = AffineTransform(
            matrix=cumulative_matrix.copy()
        )

        transformations[name] = cumulative_transform

        # STEP 4: Transform original image directly to reference
        registered_images[name] = warp(
            img,
            inverse_map=cumulative_transform,
            output_shape=ref_img.shape,
            preserve_range=True
        )

        # STEP 5: Store statistics
        results.append({
            "image": name,
            "matches": result["n_matches"],
            "filtered": result["n_filtered"],
            "inliers": result["n_inliers"],
            "rms_error": result["rms_error"],
            "status": "registered"
        })

        print(
            f"  Inliers: {result['n_inliers']}, "
            f"RMS: {result['rms_error']:.3f} pixels"
        )

        previous_name = name

    results_df = pd.DataFrame(results)

    return (
        registered_images,
        transformations,
        pairwise_transformations,
        results_df
    )


Kjører funksjonen: 

In [ ]:
registered_images, transformations, pairwise_transformations, results_df = register_all_images(
    original_images,
    reference="002",
    max_distance=20
)

display(results_df)



Sammenlikner transformasjonsmatrisene: 

In [ ]:
for name, transform in transformations.items():

    print(f"\nImage {name}")
    print(transform.params)

## Circular Hough Transform --> identifisere sirkler i ett bilde

Fungerer omtrent slik: 

1. Edge detection: Vi finner kantene i bildet, for eksempel med Canny.



2. Hough transform: Algoritmen undersøker hvilke sirkelsentre og radier som best forklarer kantene.



3. Circle detection: Vi velger de mest sannsynlige sirklene.



4. Mask generation: Vi lager en binær maske for hver identifiserte disk.

--> skal gi oss geometrisk sentrum og radius i hver disk

### Canny plot --> finner kantene på bildet

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from skimage.feature import canny
from skimage.transform import hough_circle, hough_circle_peaks
from skimage.draw import disk
from skimage.util import img_as_float

# Use reference image (0 degrees tilt)
image = img_as_float(registered_images["002"])

# Detect edges using Canny
edges = canny(image, sigma=2, low_threshold=0.04, high_threshold=0.05)

# Plot original image and detected edges
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.imshow(image, cmap="gray")
plt.title("Reference image (002)")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(edges, cmap="gray")
plt.title("Detected edges (Canny)")
plt.axis("off")

plt.tight_layout()
plt.show()

### Detekterer diskene

In [ ]:
# Possible disk radii (pixels)
hough_radii = np.arange(8, 11, 1)

# Calculate Hough transform
hough_res = hough_circle(edges, hough_radii)

# Find the strongest circle candidates
accums, cx, cy, radii = hough_circle_peaks(
    hough_res,
    hough_radii,
    total_num_peaks=68,
    min_xdistance=25,
    min_ydistance=25
)

print(f"Detected circles: {len(cx)}")

plt.figure(figsize=(8, 10))
plt.imshow(image, cmap="gray")

for x, y, r in zip(cx, cy, radii):
    circle = plt.Circle(
        (x, y),
        r,
        color="red",
        fill=False,
        linewidth=1.5
    )
    plt.gca().add_patch(circle)

plt.title(f"Detected disks: {len(cx)} / 68")
plt.axis("off")
plt.show()

### Sorterer og nummererer diskene

Indekserer hver maske på en naturlig måte, starter oppe til høyre. 

In [ ]:
# Sort circles by x-coordinate, from right to left
indices = np.argsort(-cx)

# Group circles into columns
columns = []
current_column = [indices[0]]

tolerance = 15  # Maximum x-deviation within a column (pixels)

for idx in indices[1:]:

    # Compare with mean x-position of current column
    mean_x = np.mean(cx[current_column])

    if abs(cx[idx] - mean_x) < tolerance:
        current_column.append(idx)

    else:
        columns.append(current_column)
        current_column = [idx]

columns.append(current_column)

# Sort each column from top to bottom
sorted_indices = []

for column in columns:
    column_sorted = sorted(column, key=lambda i: cy[i])
    sorted_indices.extend(column_sorted)

# Reorder detected circles
cx = cx[sorted_indices]
cy = cy[sorted_indices]
radii = radii[sorted_indices]
accums = accums[sorted_indices]

Plotter sirklene med nummerering: 

In [ ]:
plt.figure(figsize=(10, 10))
plt.imshow(image, cmap="gray")

for i, (x, y, r) in enumerate(zip(cx, cy, radii)):

    circle = plt.Circle(
        (x, y), r,
        color="red",
        fill=False,
        linewidth=1
    )
    plt.gca().add_patch(circle)

    plt.text(
        x, y, str(i + 1),
        color="yellow",
        fontsize=8,
        ha="center",
        va="center"
    )

plt.title("Disks numbered from right to left")
plt.axis("off")
plt.show()

Sammenlikner "masken" (egt bare sirkelen som omringer masken) på flere bilder for å undersøke hvor godt den treffer: 

In [ ]:
disk_number = 15
i = disk_number - 1

names = ["002", "008", "015", "022"]

x, y, r = cx[i], cy[i], radii[i]

plt.figure(figsize=(12, 3))

for j, name in enumerate(names):

    plt.subplot(1, len(names), j + 1)
    plt.imshow(registered_images[name], cmap="gray")

    circle = plt.Circle(
        (x, y), r,
        edgecolor="red",
        fill=False,
        linewidth=1
    )
    plt.gca().add_patch(circle)

    plt.xlim(x - 20, x + 20)
    plt.ylim(y + 20, y - 20)

    plt.title(name)
    plt.axis("off")

plt.tight_layout()
plt.show()

### Translasjon: phase-cross-correlation

Finjusterer nå hvor godt maskene (sirklene) treffer diskene på de ulike bildene ved å bukde phase-cross-correlation --> finner relativ translasjons-offset mellom bildene ift referansebilde

In [ ]:
from scipy.ndimage import shift
from skimage.registration import phase_cross_correlation as pcc
import numpy as np

ref_img = registered_images["002"]

shifts = {}
aligned_images = {}

for name, img in registered_images.items():

    # Calculate translation relative to reference image
    s = pcc(
        ref_img,
        img,
        normalization="phase",
        upsample_factor=100
    )[0]

    # Save translation (dy, dx)
    shifts[name] = s

    # Apply translation to the entire image
    aligned_images[name] = shift(img, s)

    print(f"{name}: dy = {s[0]:.2f}, dx = {s[1]:.2f}")


In [ ]:
names = sorted(shifts.keys(), key=int)

dy = [shifts[name][0] for name in names]
dx = [shifts[name][1] for name in names]

# Total displacement in pixels
displacement = np.sqrt(np.array(dx)**2 + np.array(dy)**2)

plt.figure(figsize=(10, 4))

plt.plot(names, displacement, "o-")
plt.axhline(1, color="red", linestyle="--", label="1 pixel")

plt.xlabel("Image")
plt.ylabel("Translation magnitude (pixels)")
plt.title("Translation after sequential affine registration")
plt.xticks(rotation=90)
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Mean translation: {np.mean(displacement):.2f} pixels")
print(f"Maximum translation: {np.max(displacement):.2f} pixels")

Ser ut som at forskyvningen øker med antall biler --> akkumuleringsfeil som følge av sekvensiell affintransformasjon? 

In [ ]:
%matplotlib widget

### Undersøker effekten av translasjon på utvalgte bilder: 

In [ ]:
disk_number = 40
i = disk_number - 1

names = ["002", "008", "015", "022"]

x, y, r = cx[i], cy[i], radii[i]

plt.figure(figsize=(12, 6))

for j, name in enumerate(names):

    for row, (images, title) in enumerate([
        (registered_images, "Before translation"),
        (aligned_images, "After translation")
    ]):

        plt.subplot(2, len(names), row * len(names) + j + 1)

        plt.imshow(images[name], cmap="gray")

        circle = plt.Circle(
            (x, y), r,
            edgecolor="red",
            fill=False,
            linewidth=1
        )
        plt.gca().add_patch(circle)

        plt.xlim(x - 20, x + 20)
        plt.ylim(y + 20, y - 20)

        plt.title(f"{name} - {title}")
        plt.axis("off")

plt.tight_layout()
plt.show()


### Lager masker

Fyller inn pixlene innenfor sirklene med 1-ere, resten blir 0-ere, markerer også pixlene som tilhører selve sirkelen --> maske i numpy array. Bruker samme maske på alle bildene, ettersom alle skal være registrert i det samme koordinatsystemet. 

In [ ]:
from skimage.draw import disk
import numpy as np

image_shape = registered_images["002"].shape

disk_masks = []

for x, y, r in zip(cx, cy, radii):

    mask = np.zeros(image_shape, dtype=bool)

    rr, cc = disk((y, x), r, shape=image_shape)
    mask[rr, cc] = True

    disk_masks.append(mask)

disk_masks_array = np.stack(disk_masks)

print(f"Created {len(disk_masks)} disk masks")

Liten kontroll --> passer maskene på bildene?

In [ ]:
disk_number = 15
i = disk_number - 1

# Choose images to compare
names = ["002", "008", "015", "022"]

x = cx[i]
y = cy[i]
mask = disk_masks[i]

plt.figure(figsize=(12, 3))

for j, name in enumerate(names):

    img = aligned_images[name]

    plt.subplot(1, len(names), j + 1)

    plt.imshow(img, cmap="gray")

    # Plot mask contour
    plt.contour(
        mask,
        levels=[0.5],
        colors="red",
        linewidths=1
    )

    # Zoom in on selected disk
    plt.xlim(x - 20, x + 20)
    plt.ylim(y + 20, y - 20)

    plt.title(name)
    plt.axis("off")

plt.tight_layout()
plt.show()

### Undersøker hvor stor effekt de ulike stegene faktisk har hatt:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

names = ["002", "008", "015", "022"]

# Bruk aligned_images hvis du har dictionary,
# eller lag den fra listen aligned
if "aligned_images" not in globals():
    aligned_images = dict(zip(registered_images.keys(), aligned))

stages = [
    (original_images, "Original"),
    (registered_images, "Affine"),
    (aligned_images, "Affine + translation")
]

plt.figure(figsize=(16, 12))

for row, (image_dict, stage_name) in enumerate(stages):

    for j, name in enumerate(names):

        img = image_dict[name]

        if not isinstance(img, np.ndarray):
            img = img.data

        plt.subplot(3, 4, row * 4 + j + 1)

        plt.imshow(img, cmap="gray")

        # Plot all 68 masks
        for mask in disk_masks:
            plt.contour(
                mask,
                levels=[0.5],
                colors="red",
                linewidths=0.7
            )

        plt.title(f"{name} - {stage_name}")
        plt.axis("off")

plt.tight_layout()
plt.show()

Zoomer inn på én og en disk

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

disk_number = 65
i = disk_number - 1

names = ["002", "008", "015", "022"]

x, y = cx[i], cy[i]
mask = disk_masks[i]

stages = [
    (original_images, "Original"),
    (registered_images, "After affine"),
    (aligned_images, "After translation")
]

plt.figure(figsize=(12, 9))

for row, (image_dict, stage_name) in enumerate(stages):
    for j, name in enumerate(names):

        img = image_dict[name]

        if not isinstance(img, np.ndarray):
            img = img.data

        plt.subplot(3, 4, row * 4 + j + 1)

        plt.imshow(img, cmap="gray")
        plt.contour(mask, levels=[0.5],
                    colors="red", linewidths=1)

        plt.xlim(x - 20, x + 20)
        plt.ylim(y + 20, y - 20)

        plt.title(f"{name} - {stage_name}")
        plt.axis("off")

plt.tight_layout()
plt.show()

### Ser på selve maskene: 

Liste med masker: 

In [ ]:
disk_masks

Maske tilhørende én diskm, som en Boolean array

In [ ]:
disk_masks[17]  # Maske for disk 18

kan få ut sentrun og radius av hver disk også:

In [ ]:
cx[17]     # x-koordinat til sentrum av disk 18
cy[17]     # y-koordinat til sentrum av disk 18
radii[17]  # Radius til disk 18

Transformasjonsmatrise til hvert bilde: 

In [ ]:
transformations["015"].params

Translasjonsskift til hvert bilde: 

In [ ]:
print(shifts["015"])

Dette kan man bruke på STEM-DPC signalet slik: 

In [ ]:
dpc_registered = warp(
    dpc_img,
    inverse_map=AffineTransform(matrix=T),
    preserve_range=True
)

dpc_aligned = shift(
    dpc_registered,
    shift_vector
)